# SCOE BAU dampening — bajar buildings de ~23.71 Mt a ~16 Mt (2050)

**Objetivo.** El BAU actual produce **23.71 MtCO2e** en 1.A.4 (Other Sectors / Buildings) en 2050. La SNBC Reference muestra ~16-17 Mt para residencial en 2050 (Figura 23, p.56 del SNBC). Hay que bajar la trayectoria moviendo los drivers de SCOE sin tocar la calibración histórica (tp 0-7).

**Diagnóstico.** En `sisepuede_raw_input_morocco_fuels.csv`:

| Driver | tp0-tp7 (calibración) | tp8-tp35 (forward) |
|---|---|---|
| `elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc` | 0.96 → -0.025 | **0.50** ← muy alto |
| `elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc` | 0.96 → -0.025 | **0.50** ← muy alto |
| `scalar_scoe_heat_energy_demand_residential` | 1.09 → 1.57 | 1.62 → 0.98 |
| `scalar_scoe_appliance_energy_demand_residential` | 1.09 → 1.57 | 1.62 → 0.98 |

**Estrategia.** Dos palancas que se complementan, sólo desde tp=8 en adelante:
1. **Bajar elasticidad residencial** de 0.50 → **0.30** (heat y appliance).
2. **Aplicar dampening multiplicativo** a los scalars residenciales: factor que rampa de 1.00 en tp=8 a **0.80** en tp=35.

Esto deja intacta la calibración histórica y conserva la forma de la trayectoria SNBC (pico hacia 2025-2030 y luego declinación), sólo con menor amplitud.

**Salida.** Nuevo CSV: `sisepuede_raw_input_morocco_fuels_scoe_bau.csv` en la misma carpeta `input_data/`.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR = REPO / 'ssp_modeling' / 'input_data'

INPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe_inen_ippu.csv'
OUTPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_inen_ippu_scoe.csv'

# --- Knobs para iterar (dial estos si la trayectoria no cae a ~16 Mt) ---
TP_CALIB_END = 7              # último tp de calibración (no tocar tp 0..7)
TP_TARGET = 35                # tp del objetivo (año 2050)

NEW_ELASTICITY_HEAT = 0.285    # bajar de 0.50 → 0.30
NEW_ELASTICITY_ELEC = 0.285    # bajar de 0.50 → 0.30

DAMPEN_START = 1.00           # factor multiplicativo en tp=8
DAMPEN_END = 0.80             # factor multiplicativo en tp=35 (= -20%)
DAMPEN_TAIL = 0.80            # mantener el factor para tp>35 (2050+)

pd.options.display.float_format = '{:.4f}'.format
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {INPUT_FILE.name}: {df.shape}  tp=[{df.time_period.min()}, {df.time_period.max()}]')

Loaded sisepuede_raw_input_morocco_fuels_scoe_inen_ippu.csv: (56, 2442)  tp=[0, 55]


## 1 · Auditoría — valores actuales de los drivers residenciales

In [14]:
DRIVERS_ELAS = [
    'elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc',
    'elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc',
]
DRIVERS_SCALAR = [
    'scalar_scoe_heat_energy_demand_residential',
    'scalar_scoe_appliance_energy_demand_residential',
]

snapshot_tps = [0, 7, 8, 15, 25, 35, 45, 55]
before = df.loc[df.time_period.isin(snapshot_tps), ['time_period', 'year'] + DRIVERS_ELAS + DRIVERS_SCALAR]
before.set_index('time_period')

,year,elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc,elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc,scalar_scoe_heat_energy_demand_residential,scalar_scoe_appliance_energy_demand_residential
time_period,,,,,
0,2015,0.9603,0.9603,1.0893,1.0893
7,2022,-0.0251,-0.0251,1.5708,1.5708
8,2023,0.3000,0.3000,1.6244,1.6244
15,2030,0.3000,0.3000,1.5401,1.5401
25,2040,0.3000,0.3000,1.4198,1.4198
35,2050,0.3000,0.3000,1.2995,1.2995
45,2060,0.3000,0.3000,1.2995,1.2995
55,2070,0.3000,0.3000,1.2995,1.2995


## 2 · Construir la trayectoria de dampening

Rampa lineal de `DAMPEN_START` en `TP_CALIB_END+1` (tp=8) a `DAMPEN_END` en `TP_TARGET` (tp=35).
Para tp > 35 se mantiene constante en `DAMPEN_TAIL` para no rebotar la trayectoria post-2050.

In [15]:
tp = df['time_period'].to_numpy()
dampen = np.ones_like(tp, dtype=float)

ramp_mask = (tp > TP_CALIB_END) & (tp <= TP_TARGET)
ramp_tp = tp[ramp_mask]
ramp_vals = np.interp(
    ramp_tp,
    [TP_CALIB_END + 1, TP_TARGET],
    [DAMPEN_START, DAMPEN_END],
)
dampen[ramp_mask] = ramp_vals
dampen[tp > TP_TARGET] = DAMPEN_TAIL

pd.DataFrame({'time_period': tp, 'year': df['year'], 'dampen_factor': dampen}).query('time_period in @snapshot_tps')

,time_period,year,dampen_factor
0,0,2015,1.0000
7,7,2022,1.0000
8,8,2023,1.0000
15,15,2030,0.9481
25,25,2040,0.8741
35,35,2050,0.8000
45,45,2060,0.8000
55,55,2070,0.8000


## 3 · Aplicar cambios

1. Bajar elasticidades residenciales para tp > 7 (heat → 0.30, elec → 0.30).
2. Multiplicar los dos scalars residenciales por el `dampen_factor`.

In [16]:
df_new = df.copy()
forward = df_new['time_period'] > TP_CALIB_END

df_new.loc[forward, 'elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc'] = NEW_ELASTICITY_HEAT
df_new.loc[forward, 'elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc'] = NEW_ELASTICITY_ELEC

for col in DRIVERS_SCALAR:
    df_new[col] = df_new[col].to_numpy() * dampen

after = df_new.loc[df_new.time_period.isin(snapshot_tps), ['time_period', 'year'] + DRIVERS_ELAS + DRIVERS_SCALAR]
after.set_index('time_period')

,year,elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc,elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc,scalar_scoe_heat_energy_demand_residential,scalar_scoe_appliance_energy_demand_residential
time_period,,,,,
0,2015,0.9603,0.9603,1.0893,1.0893
7,2022,-0.0251,-0.0251,1.5708,1.5708
8,2023,0.2850,0.2850,1.6244,1.6244
15,2030,0.2850,0.2850,1.4603,1.4603
25,2040,0.2850,0.2850,1.2410,1.2410
35,2050,0.2850,0.2850,1.0396,1.0396
45,2060,0.2850,0.2850,1.0396,1.0396
55,2070,0.2850,0.2850,1.0396,1.0396


## 4 · Verificación — diff antes/después

In [17]:
diff = (df_new[DRIVERS_ELAS + DRIVERS_SCALAR] - df[DRIVERS_ELAS + DRIVERS_SCALAR]).abs()
print('Filas con cambio en cada columna:')
print((diff > 1e-12).sum())
print()
print('tp=0..7 (calibración histórica) deben estar intactos:')
calib_diff = diff.loc[df['time_period'] <= TP_CALIB_END].max()
print(calib_diff)
assert (calib_diff < 1e-12).all(), 'Calibración histórica fue modificada — abortar.'
print('OK: calibración tp=0..7 intacta.')

Filas con cambio en cada columna:
elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc        48
elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc    48
scalar_scoe_heat_energy_demand_residential                             47
scalar_scoe_appliance_energy_demand_residential                        47
dtype: int64

tp=0..7 (calibración histórica) deben estar intactos:
elasticity_scoe_enerdem_per_hh_residential_heat_energy_to_gdppc       0.0000
elasticity_scoe_enerdem_per_hh_residential_elec_appliances_to_gdppc   0.0000
scalar_scoe_heat_energy_demand_residential                            0.0000
scalar_scoe_appliance_energy_demand_residential                       0.0000
dtype: float64
OK: calibración tp=0..7 intacta.


## 5 · Guardar nuevo CSV

In [18]:
df_new.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {OUTPUT_FILE}')
print(f'Shape: {df_new.shape}')

Wrote /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels_inen_ippu_scoe.csv
Shape: (56, 2442)


## 6 · Próximo paso

Correr el modelo con este nuevo input y verificar la trayectoria 1.A.4 en 2050:
- Si **>18 Mt**: bajar más → poner `NEW_ELASTICITY_HEAT=0.20`, `DAMPEN_END=0.70`.
- Si **<14 Mt**: subir un poco → `DAMPEN_END=0.85`.
- Si entre 15-17 Mt: 👍 conservar y mover al siguiente sector.

El comercial (`elasticity_scoe_enerdem_per_mmmgdp_commercial_municipal_*`) está en 0.0 y no aporta crecimiento al BAU. Si después de ajustar residencial la curva 1.A.4 cae demasiado, puede ser que el problema haya estado en el comercial; en ese caso revisar SNBC Fig 23 (residencial sólo ~9 Mt en 2025 ramping a ~16 en 2060) vs Fig 22 (tertiary).